## In this Project we will implement complete Medallion Architecture, We will import files from ADLS and then dump into Bronze layer. We are taking SAMPLE CUSTOMER DATA For this.  

In [ ]:
from snowflake.snowpark.context import get_active_session

from snowflake.snowpark.functions import (
        col, trim, upper, initcap, lit, to_date, when, is_null, 
        round as sf_round, sum as sf_sum, count as sf_count, 
        year, month, dayofmonth, dayofweek, dayname, monthname,
        quarter, date_part, iff, coalesce, length
)

from snowflake.snowpark.types import (
    DecimalType, 
    StringType, 
    IntegerType, 
    FloatType, 
    BooleanType, 
    DateType, 
    TimestampType,
    ArrayType,
    MapType,
    VariantType
)

session = get_active_session()

In [ ]:
# Target Databses / Schema for this Demo
session.sql("CREATE DATABASE IF NOT EXISTS YOUTUBEDEMO").collect()
session.sql("CREATE OR REPLACE SCHEMA YOUTUBEDEMO.BRONZE").collect()
session.sql("CREATE OR REPLACE SCHEMA YOUTUBEDEMO.SILVER").collect()
session.sql("CREATE OR REPLACE SCHEMA YOUTUBEDEMO.GOLD").collect()

src = "SNOWFLAKE_SAMPLE_DATA.TPCH_SF1"

In [ ]:
session.sql("""
    CREATE OR REPLACE FILE FORMAT YOUTUBEDEMO.BRONZE.CSV_FF
        TYPE = CSV
        FIELD_DELIMITER = ','
        SKIP_HEADER = 1
        FIELD_OPTIONALLY_ENCLOSED_BY = '"'
        NULL_IF = ('', 'NULL')
    """).collect()

In [ ]:
session.sql("""
    CREATE TABLE IF NOT EXISTS YOUTUBEDEMO.BRONZE.CUSTOMER_EXT (
        CUSTKEY NUMBER,
        NAME    STRING,
        ADDRESS STRING,
        NATIONKEY NUMBER,
        PHONE   STRING,
        ACCTBAL NUMBER(12,2),
        MKTSEGMENT STRING,
        COMMENT STRING
    )
    """).collect()

In [ ]:
# Storage integration --> Same like SQL query
session.sql("""
    CREATE OR REPLACE STORAGE INTEGRATION S3_TPCH_INT
        TYPE = EXTERNAL_STAGE
        STORAGE_PROVIDER = S3
        ENABLED = TRUE
        STORAGE_AWS_ROLE_ARN = 'aen:aws:iam::<account_id>:role/<snowflake-access-role>'
        STORAGE_ALLOWD_LOCATIONS = ('s3://cly-tpch-raw/customer/')
    """).collect()

# Create External Stage after authenticate in AWS IAM
session.sql("""
    CREATE OR REPLACE STAGE CLY_TPCH_DEMO.BRONZE.S3_CUSTOMER_STAGE
        STORAGE_INTEGRATION = S3_TPCH_INT
        URL = 's3://cly-tpch-raw/customer/'
        FILE_FORMAT = CLY_TPCH_DEMO.BRONZE.CSV_FF
    """).collect()

# COPY INTO COMMAND
session.sql("""
    COPY INTO CLY_TPCH_DEMO.BRONZE.CUSTOMER_EXT
    FROM @CLY_TPCH_DEMO.BRONZE.S3_CUSTOMER_STAGE
    PATTERN = '.*customer.*[.]csv'
    ON_ERROR = 'CONTINUE'
    """).collect()

In [ ]:
df_customer_raw = session.table(f"{src}.CUSTOMER")
df_orders_raw = session.table(f"{src}.ORDERS")
df_lineitem_raw = session.table(f"{src}.LINEITEM")
df_nation_raw = session.table(f"{src}.NATION")
df_region_raw = session.table(f"{src}.REGION")

print('Row Counts (Bronze):')
print('Customer :', df_customer_raw.count())
print('Orders :', df_orders_raw.count())
print('Lineitem :', df_lineitem_raw.count())
print('Nation :', df_nation_raw.count())
print('Region :', df_region_raw.count())

In [ ]:
df_customer_raw.write.save_as_table("YOUTUBEDEMO.BRONZE.CUSTOMER", mode="overwrite")
df_orders_raw.write.save_as_table("YOUTUBEDEMO.BRONZE.ORDERS", mode="overwrite")
df_lineitem_raw.write.save_as_table("YOUTUBEDEMO.BRONZE.LINEITEM", mode="overwrite")
df_nation_raw.write.save_as_table("YOUTUBEDEMO.BRONZE.NATION", mode="overwrite")
df_region_raw.write.save_as_table("YOUTUBEDEMO.BRONZE.REGION", mode="overwrite")

## NOW LET's CLEAN THE DATA ---> SILVER LAYER

In [ ]:
df_customer_silver = (
    df_customer_raw\
    .select(
    col("C_CUSTKEY").alias("CUSTOMER_KEY"),
    trim(col("C_NAME")).alias("CUSTOMER_NAME"),
    trim(col("C_ADDRESS")).alias("CUSTOMEr_ADDRESS"),
    col("C_NATIONKEY").alias("NATION_KEY"),
    trim(col("C_PHONE")).alias("CUSTOMEr_PHONE"),
    upper(trim(col("C_MKTSEGMENT"))).alias("MARKET_SEGMENT")
    )\
    .filter(col("CUSTOMER_KEY").is_not_null())
    .dropDuplicates(["CUSTOMER_KEY"])
    .withColumn(
        "MARKET_SEGMENT",
        coalesce(col("MARKET_SEGMENT"), lit("UNKNOWN"))
        )
)

df_customer_silver.write.save_as_table("YOUTUBEDEMO.SILVER.CUSTOMER_SILVER", mode = "overwrite")

In [ ]:
# Silver Nation Table
df_nation_silver = (
        df_nation_raw\
        .select(
            col("N_NATIONKEY").alias("NATION_KEY"),
            initcap(trim(col("N_NAME"))).alias("NATION_NAME"),
            col("N_REGIONKEY").alias("REGION_KEY")
        )\
        .dropDuplicates(["NATION_KEY"])
    )
    
df_nation_silver.write.save_as_table("YOUTUBEDEMO.SILVER.NATION_SILVER", mode = "overwrite")

# Sliver Region Table
df_region_silver = (
        df_region_raw\
        .select(       
            col("R_REGIONKEY").alias("REGION_KEY"),
            initcap(trim(col("R_NAME"))).alias("REGION_NAME"),
        )\
        .dropDuplicates(["REGION_KEY"])
    )

df_region_silver.write.save_as_table("YOUTUBEDEMO.SILVER.REGION_SILVER", mode = "overwrite")

In [ ]:
# SILVER ORDER TABLE
df_orders_raw\
    .select(
        col("O_ORDERKEY").alias("ORDER_KEY"),
        col("O_CUSTKEY").alias("CUSTOMER_KEY"),
        upper(trim(col("O_ORDERSTATUS"))).alias("ORDER_STATUS"),
        col("O_TOTALPRICE").cast(DecimalType(15,2)).alias("TOTAL_PRICE"),
        to_date(col("O_ORDERDATE")).alias("ORDER_DATE"),
        initcap(trim(col("O_ORDERPRIORITY"))).alias("ORDER_PRIORITY"),
        trim(col("O_CLERK")).alias("CLERK")
    )\
    .filter(col("ORDER_KEY").is_not_null())
    .dropDuplicates(["ORDER_KEY"])
    .withColumn(\
        "ORDER_STATUS",
        when(col("ORDER_STATUS") == "O", lit("OPEN"))
        .when(col("ORDER_STATUS") == "F", lit("FULFILLED"))
        .when(col("ORDER_STATUS") == "P", lit("PENDING"))
        .otherwise(col("ORDER_STATUS"))
    )